# Smart Enrichment Queue: API walkthrough

**Stop guessing which leads deserve your credits.**

Finding hundreds of scraped companies is easy. Knowing which ones are worth an enrichment credit is the hard part. This notebook is the API demonstration of that decision: rank the catalogue, read the explanation, preview a stricter profile without saving, queue only the fits, then run **simulated** enrichment.

No extra packages. Cells use the Python standard library.

## Run this demo

1. From the repo root: `npm run db:up && npm run db:migrate && npm run db:seed && npm run dev`
2. Open this file and **Run All**.
3. Optional: set `SEQ_API_BASE` if the API is not `http://localhost:4000/api/v1`.

The UI guide is http://localhost:5173. The workspace is http://localhost:5173/?view=leads.

Queue steps mutate the **one shared demo user**. Enrichment is labelled `simulated: true` and does not call a provider.

In [1]:
import json
import os
import urllib.error
import urllib.request
from pathlib import Path

BASE = os.environ.get('SEQ_API_BASE', 'http://localhost:4000/api/v1').rstrip('/')

def api(method, path, body=None, query=None):
    url = f'{BASE}{path}'
    if query:
        from urllib.parse import urlencode
        url = f'{url}?{urlencode({k: v for k, v in query.items() if v is not None})}'
    data = None if body is None else json.dumps(body).encode()
    headers = {'Accept': 'application/json'}
    if data is not None:
        headers['Content-Type'] = 'application/json'
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req) as res:
            raw = res.read()
            parsed = json.loads(raw) if raw else None
            return res.status, parsed
    except urllib.error.HTTPError as error:
        raw = error.read()
        parsed = json.loads(raw) if raw else None
        return error.code, parsed

print('API', BASE)

API http://localhost:4000/api/v1


## 1. The API is up, and enrichment is a demo

`/health/ready` includes Postgres. `/meta` advertises filter options, score thresholds, and that credits are a demo estimate.

In [2]:
print('liveness', api('GET', '/health')[1])
print('ready', api('GET', '/health/ready')[1])
meta = api('GET', '/meta')[1]
print({
    'creditsPerEnrichment': meta['creditsPerEnrichment'],
    'demo': meta.get('demo'),
    'thresholds': meta['thresholds'],
    'industries': meta['industries'][:5],
})

liveness {'status': 'ok'}
ready {'status': 'ready'}
{'creditsPerEnrichment': 1, 'demo': {'syntheticData': True, 'simulatedEnrichment': True, 'authentication': False}, 'thresholds': {'priority': {'high': 75, 'medium': 50}, 'confidence': {'high': 0.8, 'medium': 0.5}}, 'industries': ['Accounting', 'Construction', 'Education', 'Financial Services', 'HVAC Services']}


## 2. Rank the full catalogue before you spend

`GET /leads/ranked` scores every matching company against the saved ICP, then paginates. Rank numbers are global, not page-local. The summary tells you how many leads are actually worth a look.

In [3]:
ranked = api('GET', '/leads/ranked', query={'page': 1, 'pageSize': 5})[1]
print('summary', ranked['summary'])
print()
print(f"{'rank':>4}  {'score':>5}  {'priority':<14}  {'conf':<6}  company")
for lead in ranked['items']:
    score = lead['scoring']['score']
    score_s = 'n/a' if score is None else f'{score:5.0f}'
    print(f"{lead['rank']:4}  {score_s:>5}  {lead['scoring']['priority']:<14}  {lead['scoring']['confidence']:<6}  {lead['company']['name']}")

summary {'high': 25, 'medium': 50, 'low': 21, 'needs_review': 11, 'disqualified': 43, 'total': 150}

rank  score  priority        conf    company
   1    100  high            high    Cedar Industries
   2    100  high            high    Copperfield Networks
   3    100  high            high    Granite Systems
   4    100  high            high    Lumen Cloud
   5    100  high            high    Meridian Manufacturing


## 3. Unknown is not a no

Needs-review leads are missing evidence, not confirmed mismatches. The score only uses known factors. Missing fields are listed so you can look before you spend.

In [4]:
review = api('GET', '/leads/ranked', query={'priority': 'needs_review', 'pageSize': 3})[1]
print('needs_review count', review['summary']['needs_review'])
if review['items']:
    lead = review['items'][0]
    s = lead['scoring']
    print(lead['company']['name'], '| score', s['score'], '| coverage', round(s['coverage'], 2))
    print('review reasons:', s['reviewReasons'])
    print('missing factors:', s['missingFactors'])
    print('matched:')
    for factor in s['matchedFactors']:
        print(f"  {factor['label']}: {factor['explanation']} (contrib {factor['contribution']})")
else:
    print('No needs-review leads in this seed.')

needs_review count 11
Riverbend Clinics | score 94.4 | coverage 0.62
review reasons: ['Location is a strict requirement but the value is unknown']
missing factors: ['location', 'size']
matched:
  Industry fit: Healthcare Services is a target industry (contrib 44.4)
  Revenue: $5.4M revenue is within the target ($2.0M-$50.0M) (contrib 38.9)
  Data completeness: 4 of 6 key fields are available (contrib 11.1)


## 4. Preview a stricter profile without saving

`POST /scoring/preview` re-ranks with unsaved criteria. The stored ICP is unchanged. That is the same live preview the workspace uses.

In [5]:
prefs = api('GET', '/preferences')[1]
draft = json.loads(json.dumps(prefs['criteria']))
draft['location']['strict'] = True
draft['location']['values'] = ['United States']

preview = api('POST', '/scoring/preview', body={'criteria': draft, 'pageSize': 1})[1]
after = api('GET', '/preferences')[1]

print('saved location still', after['criteria']['location'])
print('preview summary', preview['summary'])
print('saved summary still', api('GET', '/leads/ranked', query={'pageSize': 1})[1]['summary'])

saved location still {'weight': 5, 'strict': True, 'values': ['United States', 'Canada']}
preview summary {'high': 20, 'medium': 43, 'low': 16, 'needs_review': 11, 'disqualified': 60, 'total': 150}
saved summary still {'high': 25, 'medium': 50, 'low': 21, 'needs_review': 11, 'disqualified': 43, 'total': 150}


## 5. Queue only the fits

`POST /queue/items` accepts up to 100 ids and returns `added`, `duplicate`, or `not_found` per id. The unique constraint is `(user_id, company_id)`, so a second add is a duplicate, not a silent skip.

In [6]:
fresh = []
page = 1
while len(fresh) < 2 and page <= 6:
    high = api('GET', '/leads/ranked', query={'priority': 'high', 'page': page, 'pageSize': 25})[1]
    fresh.extend(lead['company']['id'] for lead in high['items'] if lead['queueStatus'] is None)
    if not high['items']:
        break
    page += 1
fresh = fresh[:2]
if len(fresh) < 2:
    raise SystemExit('Need two unqueued high-fit leads. Clear queue items or reseed, then rerun.')

status, added = api('POST', '/queue/items', body={'companyIds': fresh})
print('status', status)
print(added)

again = api('POST', '/queue/items', body={'companyIds': fresh[:1]})[1]
print('second add of the same id', again)

queue = api('GET', '/queue', query={'status': 'queued', 'pageSize': 5})[1]
print('queue summary', queue['summary'])

status 201
{'results': [{'companyId': '61ee9586-4651-4d45-9072-0aa6f2b7813b', 'outcome': 'added'}, {'companyId': '4d31a73e-1aaa-4c29-8919-036b34e72ac6', 'outcome': 'added'}], 'added': 2, 'duplicates': 0, 'notFound': 0}
second add of the same id {'results': [{'companyId': '61ee9586-4651-4d45-9072-0aa6f2b7813b', 'outcome': 'duplicate'}], 'added': 0, 'duplicates': 1, 'notFound': 0}
queue summary {'byStatus': {'queued': 2, 'enriched': 25, 'failed': 1, 'skipped': 0}, 'total': 28, 'plannedCredits': 2, 'creditsUsed': 25, 'creditsPerEnrichment': 1, 'simulated': True}


## 6. Simulated enrichment, and credits only when it works

`POST /queue/enrich` can take `itemIds` so this cell only touches what we just queued. A lead with no website domain fails, because a real provider would have nothing to match. Failed rows do not consume a credit. The payload always includes `simulated: true`.

In [7]:
queued = api('GET', '/queue', query={'status': 'queued', 'pageSize': 25})[1]
ours = [item['id'] for item in queued['items'] if item['company']['id'] in fresh]
status, enrich = api('POST', '/queue/enrich', body={'itemIds': ours})
print('status', status)
print(enrich)
assert enrich['simulated'] is True

after = api('GET', '/queue', query={'pageSize': 100})[1]
print('credits used', after['summary']['creditsUsed'], 'of', after['summary']['creditsPerEnrichment'], 'per success')
print(f"{'status':<10}  {'credits':>7}  company")
ours_set = set(ours)
for item in after['items']:
    if item['id'] in ours_set:
        print(f"{item['status']:<10}  {item['estimatedCredits']:7}  {item['company']['name']}  {item['statusReason'] or ''}")

status 200
{'enriched': 2, 'failed': 0, 'creditsUsed': 2, 'simulated': True}
credits used 27 of 1 per success
status      credits  company
enriched          1  Granite Clinics  Simulated enrichment (no provider called)
enriched          1  Ironwood Manufacturing  Simulated enrichment (no provider called)


## 7. Validation errors are explicit

Every error uses `{ error: { code, message, requestId } }`. Unknown fields are rejected. Invalid queue transitions return 409.

In [8]:
print('bad uuid', api('POST', '/queue/items', body={'companyIds': ['not-a-uuid']})[0:2])
print('unknown field', api('PATCH', f"/queue/items/{ours[0]}", body={'status': 'skipped', 'extra': True})[0:2])

bad uuid (400, {'error': {'code': 'VALIDATION_ERROR', 'message': 'The request contains invalid fields.', 'requestId': 'dbcae4d6-dbc8-4e21-99d4-a8c5d38f59ea', 'details': [{'path': 'companyIds.0', 'message': 'Invalid UUID'}]}})
unknown field (400, {'error': {'code': 'VALIDATION_ERROR', 'message': 'The request contains invalid fields.', 'requestId': '7c52dbec-1df3-4c4a-804d-61dfa016df23', 'details': [{'path': '', 'message': 'Unrecognized key: "extra"'}]}})


## What this proved

- Rank the whole filtered set before you spend a credit.
- Explanations come from the same factors as the score. Unknown data is listed, not scored as zero.
- Preview does not write the saved ICP.
- The queue reports duplicates. Enrichment is simulated and only successful rows use a credit.

UI walkthrough: http://localhost:5173 then http://localhost:5173/?view=leads.